# 01 · Explore the NHATS Data

This notebook is a compact, aggregate-only tour of the cleaned NHATS panel used by Willow. It checks the source files,
describes who enters and leaves the analytic sample, and shows whether recent functional changes are meaningfully tied
to newly needing help one year later.

Run after installing the project kernel with `pip install -e ".[dev]"`. The first run builds and caches
`data/clean/panel.parquet`; keep that respondent-level cache local.


In [ ]:
import sys
from pathlib import Path

for root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (root / "src" / "willow").exists():
        sys.path.insert(0, str(root / "src"))
        break
for name in list(sys.modules):
    if name == "willow" or name.startswith("willow."):
        del sys.modules[name]

import pandas as pd
import matplotlib.pyplot as plt

from willow import nhats, explore as ex
from willow.data import DEFAULT_NHATS, load_panel
from willow.features import build_windows
from willow.schema import ACTIVITIES, ACTIVITY_GROUPS, CHANGE_GROUPS, HELP_COLS, DIFF_COLS, DEVICE_COLS, ITEM_LABELS

NHATS_DIR = DEFAULT_NHATS
cfg = nhats.load_config()
pd.set_option("display.max_columns", 40, "display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
YEAR = {r: 2010 + r for r in range(1, 30)}


## 1. Source Files

Confirm the expected NHATS files and variable names are present. Missing sex and race/ethnicity after cohort-entry
rounds is expected because the loader carries those time-invariant fields forward.


In [ ]:
inv = nhats.file_inventory(NHATS_DIR, cfg).assign(year=lambda d: d["round"].map(YEAR))
chk = nhats.inspect(NHATS_DIR, cfg)
print(f"{chk.found.mean():.1%} of configured (variable, round) pairs found")
display(inv.set_index(["round", "year"]))
chk[~chk.found].groupby("canonical")["round"].apply(list).rename("rounds where missing").to_frame()


## 2. Sample And Attrition

NHATS begins in 2011 and replenishes the sample in 2015 and 2022. The table and plot below show the sample at entry,
community interviews over time, and what happens by the next round for people living in the community.


In [ ]:
panel = load_panel(NHATS_DIR)
comm = panel[panel.status == "community"]
print(f"{panel.person_id.nunique():,} sample people; {len(panel):,} person-rounds; rounds {panel['round'].min()}-{panel['round'].max()}")
display(ex.counts(panel.drop_duplicates("person_id"), "cohort").rename(index=lambda r: f"entered round {int(r)} ({YEAR[int(r)]})"))
display(ex.counts(panel, "round", "status"))

ax = comm.groupby(["round", "cohort"]).size().unstack().rename(index=YEAR).plot(marker="o", figsize=(8, 3.5))
ax.set_title("Community interviews by cohort", loc="left")
ax.legend(title="entry round", frameon=False)
plt.tight_layout()


In [ ]:
trans = ex.transition_table(panel)
display(trans.rename(index=lambda r: f"{r}->{r + 1}"))
ex.plot_lines(trans.T.loc[["deceased", "nursing_home", "residential_care", "not interviewed"]],
              "Next-round status (% of community residents in round t)")


## 3. Everyday Activities

These are the seven activities Willow models. Weighted percentages approximate the US population age 65+ living in the
community; the grey band marks the 2020-2021 pandemic rounds.


In [ ]:
nice = {c: c.rsplit("_", 1)[0].replace("_", " ") for c in HELP_COLS + DIFF_COLS + DEVICE_COLS}
for cols, title in [(HELP_COLS, "Receives help from a person"), (DIFF_COLS, "Difficulty doing it alone"),
                    (DEVICE_COLS, "Uses a device")]:
    ex.plot_lines(ex.table_by(comm, cols, weight="weight"), f"{title} (% of community residents, weighted)", labels=nice)

display(ex.latest_activity_table(panel, ACTIVITIES))


## 4. At-Risk Windows And Outcome

An at-risk window is a person living in the community at `t-1` and `t`, receiving no help at `t`. The main outcome is
newly receiving help with any of the seven activities at `t+1`.


In [ ]:
windows = build_windows(panel)
print(f"{len(windows):,} at-risk windows from {windows.person_id.nunique():,} people")
display(ex.counts(windows, "followup"))
ex.outcome_table(windows)


## 5. Changes Before New Help

The descriptive signal is whether decline from `t-1` to `t` precedes needing help at `t+1`. Ratios above 1 mean the
change group is associated with higher next-year help risk.


In [ ]:
groups = ACTIVITY_GROUPS + ["Recent fall", "Hospital stay", "Memory getting worse (self-report)", "Household tasks getting harder"]
display(ex.change_risk_table(windows, groups))

declines = ex.decline_count_table(windows)
ax = declines["new help"].plot.bar(figsize=(6, 3.2), rot=0, color="#2f6fb5", title="New help by number of areas that got worse")
ax.set_ylabel("%")
plt.tight_layout()
declines


## 6. Fairness Groups And Pandemic Checks

Group sizes and event rates show where later fairness estimates will be stable. Pandemic-era changes are checked
separately because round 10 was by phone and skipped performance tests.


In [ ]:
display(ex.at_risk_group_table(windows))

key = ["go_outside_less", "go_outside_wout", "go_outside_diff", "fell_last_year", "hospital_stay", "low_energy"]
ex.plot_lines(ex.table_by(comm, key, weight="weight"), "Selected items around the pandemic (weighted %)", labels=ITEM_LABELS)
ex.numeric_summary(panel, ["walk_time", "chair_time", "grip_max"]).assign(note="performance tests not given in 2020")


## Takeaways To Carry Forward

- The outcome is uncommon enough to require calibrated risk modelling rather than simple rules.
- Activity difficulty and device use appear before help, especially for going outside and bathing.
- Attrition, small subgroup counts, and pandemic data collection changes need explicit checks in modelling.
